In [4]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm

import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle

from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score
import copy

from src.train import trainModel
#from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn
from src.dataloader import getData,spliceDataset,h5pyDataset,getDataPointList,getDataPointListFull,DataPointFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer, SpliceAI_10K
from src.evaluation_metrics import print_topl_statistics,cross_entropy_2d
from src.gpu_metrics import run_bootstrap, calculate_ap, calculate_topk

In [5]:
data_dir = '../Data'

CL_max=40000
# Maximum nucleotide context length (CL_max/2 on either side of the 
# position of interest)
# CL_max should be an even number
SL=5000

BATCH_SIZE = 16

setType = 'test'
annotation, transcriptToLabel, seqData = getData(data_dir, setType)

In [16]:
train_gene, test_gene = train_test_split(annotation['gene'].drop_duplicates(),test_size=.03,random_state=435)
annotation_train = annotation[annotation['gene'].isin(train_gene)]
annotation_test = annotation[annotation['gene'].isin(test_gene)]

In [ ]:
### TESTING SPLICEFORMER

temp = 1
n_models = 10
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model_m = SpliceFormer(CL_max,bn_momentum=0.01/1,depth=4,heads=4,n_transformer_blocks=2,determenistic=True)
model_m.apply(keras_init)
model_m = model_m.to(device)

if torch.cuda.device_count() > 1:
    model_m = nn.DataParallel(model_m)
model_m = nn.DataParallel(model_m)
output_class_labels = ['Null', 'Acceptor', 'Donor']

#for output_class in [1,2]:
models = [copy.deepcopy(model_m) for i in range(n_models)]
[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_171022_{}'.format(i),map_location=device)) for i,model in enumerate(models)]
#nr = [0,2,3]
#[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_201221_{}'.format(nr[i]))) for i,model in enumerate(models)]
#chunkSize = num_idx/10
for model in models:
    model.eval()

Y_true_acceptor, Y_pred_acceptor = [],[]
Y_true_donor, Y_pred_donor = [],[]
test_dataset = spliceDataset(getDataPointListFull(annotation_test,transcriptToLabel,SL,CL_max,shift=SL))
test_dataset.seqData = seqData
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


#targets_list = []
#outputs_list = []
ce_2d = []
for (batch_features ,targets) in tqdm(test_loader):
    batch_features = batch_features.type(torch.FloatTensor).to(device)
    targets = targets.to(torch.float32).to("mps")[:,:,CL_max//2:-CL_max//2]
    outputs = ([models[i](batch_features)[0].detach() for i in range(n_models)])
    #outputs = (outputs[0]+outputs[1]+outputs[2]+outputs[3]+outputs[4])/n_models
    outputs = torch.stack(outputs)
    outputs = torch.mean(outputs,dim=0)
    #outputs = odds_gmean(outputs)
    #targets_list.extend(targets.unsqueeze(0))
    #outputs_list.extend(outputs.unsqueeze(0))

    targets = torch.transpose(targets,1,2).cpu().numpy()
    outputs = torch.transpose(outputs,1,2).cpu().numpy()
    ce_2d.append(cross_entropy_2d(targets,outputs))

    is_expr = (targets.sum(axis=(1,2)) >= 1)
    Y_true_acceptor.extend(targets[is_expr, :, 1].flatten())
    Y_true_donor.extend(targets[is_expr, :, 2].flatten())
    Y_pred_acceptor.extend(outputs[is_expr, :, 1].flatten())
    Y_pred_donor.extend(outputs[is_expr, :, 2].flatten())


In [ ]:
mean_ce = np.mean(ce_2d)
print('Cross entropy = {}'.format(mean_ce))
Y_true_acceptor, Y_pred_acceptor,Y_true_donor, Y_pred_donor = np.array(Y_true_acceptor), np.array(Y_pred_acceptor),np.array(Y_true_donor), np.array(Y_pred_donor)
print("\n\033[1m{}:\033[0m".format('Acceptor'))
acceptor_val_results = print_topl_statistics(Y_true_acceptor, Y_pred_acceptor)
print("\n\033[1m{}:\033[0m".format('Donor'))
donor_val_results =print_topl_statistics(Y_true_donor, Y_pred_donor)

In [ ]:
df = pd.DataFrame({'Y_true_acceptor':Y_true_acceptor,'Y_pred_acceptor':Y_pred_acceptor,'Y_true_donor':Y_true_donor,'Y_pred_donor':Y_pred_donor})
df.to_csv('../Data/transformer_40k_test_subset_predictions_180625.gz',index=False)

df = pd.read_csv('../Data/transformer_40k_test_subset_predictions_180625.gz')


In [ ]:
Y_true_acceptor = torch.as_tensor(df['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(df['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(df['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(df['Y_pred_donor'].values, dtype=torch.float32).to(device)

# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

In [18]:
### TESTING SPLICEAI MODEL

temp = 1
n_models = 10
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model_m = SpliceAI_10K(CL_max)
model_m.apply(keras_init)
model_m = model_m.to(device)

if torch.cuda.device_count() > 1:
    model_m = nn.DataParallel(model_m)
model_m = nn.DataParallel(model_m)

output_class_labels = ['Null', 'Acceptor', 'Donor']

#for output_class in [1,2]:
models = [copy.deepcopy(model_m) for i in range(n_models)]
[model.load_state_dict(torch.load('../Results/PyTorch_Models/spliceai_encoder_10k_071122_{}'.format(i), map_location=device)) for i,model in enumerate(models)]

for model in models:
    model.eval()
    
#nr = [0,2,3]
#[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_201221_{}'.format(nr[i]))) for i,model in enumerate(models)]
#chunkSize = num_idx/10


Y_true_acceptor, Y_pred_acceptor = [],[]
Y_true_donor, Y_pred_donor = [],[]
ce_2d = []

test_dataset = spliceDataset(getDataPointListFull(annotation_test,transcriptToLabel,SL,CL_max,shift=SL))
test_dataset.seqData = seqData
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


#targets_list = []
#outputs_list = []
ce_2d = []
for (batch_features ,targets) in tqdm(test_loader):
    batch_features = batch_features.type(torch.FloatTensor).to(device)
    targets = targets.to(torch.float32).to("mps")[:,:,CL_max//2:-CL_max//2]
    outputs = ([models[i](batch_features).detach() for i in range(n_models)])
    #outputs = (outputs[0]+outputs[1]+outputs[2]+outputs[3]+outputs[4])/n_models
    #outputs = odds_gmean(torch.stack(outputs),n_models)
    outputs = torch.mean(torch.stack(outputs),dim=0)
    #targets_list.extend(targets.unsqueeze(0))
    #outputs_list.extend(outputs.unsqueeze(0))

    targets = torch.transpose(targets,1,2).cpu().numpy()
    outputs = torch.transpose(outputs,1,2).cpu().numpy()
    ce_2d.append(cross_entropy_2d(targets,outputs))

    is_expr = (targets.sum(axis=(1,2)) >= 1)
    Y_true_acceptor.extend(targets[is_expr, :, 1].flatten())
    Y_true_donor.extend(targets[is_expr, :, 2].flatten())
    Y_pred_acceptor.extend(outputs[is_expr, :, 1].flatten())
    Y_pred_donor.extend(outputs[is_expr, :, 2].flatten())

100%|██████████| 230/230 [15:42<00:00,  4.10s/it]


In [19]:
mean_ce = np.mean(ce_2d)
print('Cross entropy = {}'.format(mean_ce))
Y_true_acceptor, Y_pred_acceptor,Y_true_donor, Y_pred_donor = np.array(Y_true_acceptor), np.array(Y_pred_acceptor),np.array(Y_true_donor), np.array(Y_pred_donor)
print("\n\033[1m{}:\033[0m".format('Acceptor'))
acceptor_val_results = print_topl_statistics(Y_true_acceptor, Y_pred_acceptor)
print("\n\033[1m{}:\033[0m".format('Donor'))
donor_val_results =print_topl_statistics(Y_true_donor, Y_pred_donor)

Cross entropy = 5.48219213669654e-05

Acceptor:
0.9992	0.9692	0.9977	0.9985	0.9935	0.9835	0.6274	0.0023	0.0003	2518	2598.0	2598

Donor:
0.9985	0.9688	0.9977	0.9977	0.9923	0.9845	0.6737	0.0018	0.0002	2517	2598.0	2598


In [20]:
df = pd.DataFrame({'Y_true_acceptor':Y_true_acceptor,'Y_pred_acceptor':Y_pred_acceptor,'Y_true_donor':Y_true_donor,'Y_pred_donor':Y_pred_donor})
df.to_csv('../Data/spliceai_10k_test_subset_predictions_190625.gz',index=False)

In [21]:
df = pd.read_csv('../Data/spliceai_10k_test_subset_predictions_190625.gz')
df

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,8.033060e-06,0.0,1.898466e-03
1,0.0,2.216914e-06,0.0,2.066190e-06
2,0.0,2.047246e-07,0.0,2.284577e-06
3,0.0,1.068288e-07,0.0,2.544122e-06
4,0.0,2.666359e-07,0.0,9.676069e-07
...,...,...,...,...
18364995,0.0,8.136786e-05,0.0,4.621359e-05
18364996,0.0,8.136780e-05,0.0,4.621361e-05
18364997,0.0,8.136788e-05,0.0,4.621367e-05
18364998,0.0,8.136781e-05,0.0,4.621358e-05


In [22]:
Y_true_acceptor = torch.as_tensor(df['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(df['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(df['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(df['Y_pred_donor'].values, dtype=torch.float32).to(device)

# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.9929
Top-k Accuracy: 0.9257


In [15]:
print(annotation)

                                                   name chrom strand  \
0     FO538757.2---ENSG00000279928.1---ENST000006244...  chr1      +   
1     NOC2L---ENSG00000188976.10---ENST00000327044.6...  chr1      -   
2     KLHL17---ENSG00000187961.13---ENST00000338591....  chr1      +   
3     PLEKHN1---ENSG00000187583.10---ENST00000379407...  chr1      +   
4     PLEKHN1---ENSG00000187583.10---ENST00000379410...  chr1      +   
...                                                 ...   ...    ...   
8950  DPH7---ENSG00000148399.12---ENST00000277540.6-...  chr9      -   
8951  ZMYND19---ENSG00000165724.5---ENST00000298585....  chr9      -   
8952  ARRDC1---ENSG00000197070.13---ENST00000371421....  chr9      +   
8953  EHMT1---ENSG00000181090.18---ENST00000462484.5...  chr9      +   
8954  CACNA1B---ENSG00000148408.12---ENST00000277549...  chr9      +   

       tx_start     tx_end       transcript             gene  
0        182393     184158  ENST00000624431  ENSG00000279928  
1        